# Agent 工具后置条件：权威回读与 State Diff 验证

**面试问题：工具返回 HTTP 200 后，Agent 为什么仍不能直接告诉用户“操作成功”？**

## 回答主线

1. 传输成功只说明服务返回响应，不证明业务状态满足用户目标。
2. 每个写工具都应声明前置条件、期望 state diff 和允许的附带变化。
3. 执行后从权威数据源回读并比较 before/after，才能识别 no-op、写错对象和部分成功。
4. 重试必须复用幂等键，避免验证超时导致二次副作用。
5. 最终一致系统要做有界退避，区分尚未可见与真正失败。
6. Agent 回复只能引用已验证的后置条件证据。

## 真实案例

六个订单工具调用都返回 200，但包含正确取消、无操作取消、改错订单、退款部分成功、地址最终一致延迟和只读查询。我们构造权威数据库、执行账本和后置条件 DSL，比较“信 HTTP 状态”与 state diff，并复现立即回读误判。案例使用仓库内生成的离线脱敏小数据，输出用于学习机制，不代表生产性能。

### 输入预览：六个工具调用及期望后置条件

In [1]:
from copy import deepcopy  # 导入深拷贝以隔离每个工具案例。

initial_db = {  # 构造三个订单的权威初始状态。
    "O1": {"status": "paid", "refunded": False, "address": "上海"},  # 正常可取消订单。
    "O2": {"status": "paid", "refunded": False, "address": "北京"},  # 用于写错对象案例。
    "O3": {"status": "shipped", "refunded": False, "address": "深圳"},  # 已发货订单。
}  # 完成初态。
calls = [  # 构造六个均返回 HTTP 200 的工具行为。
    {"id": "P1", "tool": "cancel", "target": "O1", "behavior": "correct", "expected": {"status": "cancelled"}},  # 正确取消。
    {"id": "P2", "tool": "cancel", "target": "O1", "behavior": "noop", "expected": {"status": "cancelled"}},  # 服务吞掉写入却返回成功。
    {"id": "P3", "tool": "address", "target": "O1", "behavior": "wrong-object", "expected": {"address": "杭州"}},  # 错把 O2 地址改为杭州。
    {"id": "P4", "tool": "refund", "target": "O1", "behavior": "partial", "expected": {"status": "cancelled", "refunded": True}},  # 只退款未取消形成部分成功。
    {"id": "P5", "tool": "address", "target": "O1", "behavior": "eventual", "expected": {"address": "成都"}},  # 地址更新延迟两个读周期可见。
    {"id": "P6", "tool": "read", "target": "O3", "behavior": "readonly", "expected": {"status": "shipped"}},  # 只读查询无副作用。
]  # 完成后置条件矩阵。
print("案例  tool     target  behavior      expected")  # 输出调用表头。
for call in calls:  # 逐调用展示目标和期望状态。
    print(f"{call['id']}   {call['tool']:<8} {call['target']:<6} {call['behavior']:<13} {call['expected']}")  # 展示六种业务语义。

案例  tool     target  behavior      expected
P1   cancel   O1     correct       {'status': 'cancelled'}
P2   cancel   O1     noop          {'status': 'cancelled'}
P3   address  O1     wrong-object  {'address': '杭州'}
P4   refund   O1     partial       {'status': 'cancelled', 'refunded': True}
P5   address  O1     eventual      {'address': '成都'}
P6   read     O3     readonly      {'status': 'shipped'}


## Baseline 基线：HTTP 200 就回复成功

In [2]:
baseline_rows = []  # 收集传输层判断。
for call in calls:  # 所有教学工具都模拟 HTTP 200。
    response = {"status_code": 200, "request_id": f"req-{call['id']}"}  # 构造网关响应。
    claimed_success = response["status_code"] == 200  # 错误地把传输成功当业务成功。
    baseline_rows.append({"id": call["id"], "claimed_success": claimed_success, "response": response})  # 保存基线结论。
print("案例  HTTP  Agent声称成功")  # 输出基线表头。
for row in baseline_rows:  # 逐案例展示全部被宣告成功。
    print(f"{row['id']}    {row['response']['status_code']}   {row['claimed_success']}")  # 暴露无法区分 no-op 和错对象。

案例  HTTP  Agent声称成功
P1    200   True
P2    200   True
P3    200   True
P4    200   True
P5    200   True
P6    200   True


### 核心实现：执行、副作用账本与期望 State Diff

In [3]:
def execute_tool(call):  # 在隔离数据库上模拟不同服务行为。
    database = deepcopy(initial_db)  # 为当前案例创建全新权威状态。
    before = deepcopy(database)  # 保存完整执行前快照。
    pending = []  # 保存最终一致系统尚未可见的写入。
    if call["behavior"] == "correct":  # 正确修改目标订单。
        database[call["target"]]["status"] = "cancelled"  # 写入期望状态。
    elif call["behavior"] == "noop":  # 服务返回成功但不产生副作用。
        pass  # 保持数据库完全不变。
    elif call["behavior"] == "wrong-object":  # 服务 Bug 写错资源。
        database["O2"]["address"] = "杭州"  # 修改非目标订单。
    elif call["behavior"] == "partial":  # 只完成退款子步骤。
        database[call["target"]]["refunded"] = True  # 标记已退款但未取消。
    elif call["behavior"] == "eventual":  # 写入先进入异步复制队列。
        pending.append({"visible_after": 2, "target": call["target"], "field": "address", "value": "成都"})  # 两次回读后才可见。
    return database, before, pending, {"status_code": 200, "idempotency_key": f"idem-{call['id']}"}  # 返回权威状态、待提交事件和响应。

def state_diff(before, after):  # 计算订单级字段变化。
    differences = []  # 收集每个变更字段。
    for order_id in sorted(before):  # 逐订单比较状态。
        for field in sorted(before[order_id]):  # 逐字段比较值。
            if before[order_id][field] != after[order_id][field]:  # 当前字段发生变化。
                differences.append({"order": order_id, "field": field, "before": before[order_id][field], "after": after[order_id][field]})  # 保存权威差异。
    return differences  # 返回完整 state diff。

def verify_postcondition(call, before, after):  # 检查目标字段和意外跨对象变化。
    target_state = after[call["target"]]  # 读取权威目标对象终态。
    expected_ok = all(target_state.get(field) == value for field, value in call["expected"].items())  # 检查所有期望字段。
    differences = state_diff(before, after)  # 计算实际副作用。
    unexpected_objects = sorted({item["order"] for item in differences if item["order"] != call["target"]})  # 找出写到其他资源的副作用。
    return {"verified": expected_ok and not unexpected_objects, "expected_ok": expected_ok, "unexpected_objects": unexpected_objects, "diff": differences}  # 返回后置条件证据。

demo_db, demo_before, demo_pending, demo_response = execute_tool(calls[2])  # 执行写错对象 P3。
demo_verification = verify_postcondition(calls[2], demo_before, demo_db)  # 权威回读并验证。
print("P3 HTTP响应：", demo_response)  # 展示传输层看似成功。
print("P3 State Diff：", demo_verification)  # 展示 O2 被错误修改且 O1 未满足期望。

P3 HTTP响应： {'status_code': 200, 'idempotency_key': 'idem-P3'}
P3 State Diff： {'verified': False, 'expected_ok': False, 'unexpected_objects': ['O2'], 'diff': [{'order': 'O2', 'field': 'address', 'before': '北京', 'after': '杭州'}]}


## 结果解读：逐案例比较声称成功与已验证成功

In [4]:
verification_rows = []  # 收集六个调用的首次回读结果。
execution_states = []  # 保存后续最终一致案例所需状态。
for call in calls:  # 逐工具调用运行隔离执行。
    database, before, pending, response = execute_tool(call)  # 获取副作用和响应。
    verification = verify_postcondition(call, before, database)  # 立即从权威库验证。
    verification_rows.append({"id": call["id"], **verification})  # 保存结构化结论。
    execution_states.append({"database": database, "before": before, "pending": pending, "response": response})  # 保存执行环境。
print("案例  HTTP声称  expected_ok  意外对象  首次验证  diff")  # 输出同口径结果表头。
for baseline, row in zip(baseline_rows, verification_rows):  # 对齐传输和业务验证。
    print(f"{row['id']}    {str(baseline['claimed_success']):<8} {str(row['expected_ok']):<11} {str(row['unexpected_objects']):<9} {str(row['verified']):<8} {row['diff']}")  # 展示 no-op、错对象、部分成功。
verified_count = sum(row["verified"] for row in verification_rows)  # 统计首次回读真正通过数量。
print(f"HTTP成功=6/6，首次后置条件通过={verified_count}/6")  # 量化传输层误报。
print("解读：P6 只读查询没有 diff 但期望状态成立；写工具则必须同时满足目标字段且无跨对象副作用。")  # 解释只读与写入差异。

案例  HTTP声称  expected_ok  意外对象  首次验证  diff
P1    True     True        []        True     [{'order': 'O1', 'field': 'status', 'before': 'paid', 'after': 'cancelled'}]
P2    True     False       []        False    []
P3    True     False       ['O2']    False    [{'order': 'O2', 'field': 'address', 'before': '北京', 'after': '杭州'}]
P4    True     False       []        False    [{'order': 'O1', 'field': 'refunded', 'before': False, 'after': True}]
P5    True     False       []        False    []
P6    True     True        []        True     []
HTTP成功=6/6，首次后置条件通过=2/6
解读：P6 只读查询没有 diff 但期望状态成立；写工具则必须同时满足目标字段且无跨对象副作用。


## 失败案例：最终一致写入被立即回读误判失败

In [5]:
eventual_state = execution_states[4]  # 读取 P5 的数据库和待提交事件。
immediate_result = verification_rows[4]  # 获取第一次回读结论。
poll_ledger = []  # 保存有界退避的每次权威回读。
for read_attempt in range(1, 4):  # 最多执行三次回读。
    for event in list(eventual_state["pending"]):  # 检查尚未可见的复制事件。
        if read_attempt >= event["visible_after"]:  # 当前重试次数达到可见门槛。
            eventual_state["database"][event["target"]][event["field"]] = event["value"]  # 应用最终一致写入。
            eventual_state["pending"].remove(event)  # 从待提交队列移除。
    result = verify_postcondition(calls[4], eventual_state["before"], eventual_state["database"])  # 每次从权威状态重新验证。
    poll_ledger.append({"attempt": read_attempt, "verified": result["verified"], "state": deepcopy(eventual_state["database"]["O1"]), "idempotency_key": eventual_state["response"]["idempotency_key"]})  # 保存回读证据和同一幂等键。
    if result["verified"]:  # 后置条件已经可见。
        break  # 停止继续轮询。
print("P5 立即回读：", immediate_result)  # 展示首次 no diff 并不代表写失败。
print("P5 有界回读账本：", poll_ledger)  # 展示第二次读取地址变为成都。
print("修正策略：工具声明 consistency SLA；在期限内用同一 idempotency key 轮询权威状态，超时后标记 unknown 而不是重新执行写操作。")  # 总结最终一致处理。

P5 立即回读： {'id': 'P5', 'verified': False, 'expected_ok': False, 'unexpected_objects': [], 'diff': []}
P5 有界回读账本： [{'attempt': 1, 'verified': False, 'state': {'status': 'paid', 'refunded': False, 'address': '上海'}, 'idempotency_key': 'idem-P5'}, {'attempt': 2, 'verified': True, 'state': {'status': 'paid', 'refunded': False, 'address': '成都'}, 'idempotency_key': 'idem-P5'}]
修正策略：工具声明 consistency SLA；在期限内用同一 idempotency key 轮询权威状态，超时后标记 unknown 而不是重新执行写操作。


### 生产边界与回复证据

In [6]:
reply_evidence = {"call_id": "P1", "tool_request": "req-P1", "idempotency_key": "idem-P1", "verified": verification_rows[0]["verified"], "postcondition": calls[0]["expected"], "state_diff": verification_rows[0]["diff"], "source": "orders-primary"}  # 构造可供 Agent 回复引用的证据。
print("回复证据：", reply_evidence)  # 展示“已取消”必须绑定权威 state diff。
print("生产替换点：真实系统需要 JSON Schema 后置条件、事务版本/ETag、事件一致性 SLA、幂等存储、补偿工作流、审计和敏感字段脱敏。")  # 明确内存数据库边界。

回复证据： {'call_id': 'P1', 'tool_request': 'req-P1', 'idempotency_key': 'idem-P1', 'verified': True, 'postcondition': {'status': 'cancelled'}, 'state_diff': [{'order': 'O1', 'field': 'status', 'before': 'paid', 'after': 'cancelled'}], 'source': 'orders-primary'}
生产替换点：真实系统需要 JSON Schema 后置条件、事务版本/ETag、事件一致性 SLA、幂等存储、补偿工作流、审计和敏感字段脱敏。


## 回归测试：最后只保护 No-op、错对象、部分成功与最终一致

In [7]:
assert verification_rows[0]["verified"] and verification_rows[0]["diff"][0]["after"] == "cancelled"  # 验证正确取消有权威 diff。
assert not verification_rows[1]["verified"] and verification_rows[1]["diff"] == []  # 验证 HTTP 200 的 no-op 被识别。
assert not verification_rows[2]["verified"] and verification_rows[2]["unexpected_objects"] == ["O2"]  # 验证写错资源被识别。
assert not verification_rows[3]["verified"] and verification_rows[3]["expected_ok"] is False  # 验证部分成功未满足完整后置条件。
assert not immediate_result["verified"] and poll_ledger[-1]["verified"] and len({row["idempotency_key"] for row in poll_ledger}) == 1  # 验证最终一致通过有界回读且不换幂等键。
print("回归测试通过：正确 Diff、No-op、错对象、部分成功和最终一致回读均成立。")  # 用少量断言总结工具后置条件合同。

回归测试通过：正确 Diff、No-op、错对象、部分成功和最终一致回读均成立。
